# Cai dat thuat toan Canny Edge Detection tu dau

**Muc tieu:** Tu cai dat 4 buoc cua thuat toan Canny (Gaussian Blur -> Sobel Gradient -> Non-Maximum Suppression -> Hysteresis Thresholding), sau do so sanh voi ham `cv2.Canny()` co san cua OpenCV.

**Trang thai:** Dang lam / Draft

**Input:** anh dat trong thu muc `input/` (mac dinh: `input/circle.jpg`)

**Output:** hinh so sanh 6 buoc duoc luu vao `output/canny_steps.png`


In [ ]:
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os

INPUT_PATH = os.path.join('..', 'input', 'circle.jpg')
OUTPUT_PATH = os.path.join('..', 'output', 'canny_steps.png')

## Buoc 0: Doc anh va chuyen sang anh xam

In [ ]:
image = cv2.imread(INPUT_PATH)
if image is None:
    raise FileNotFoundError(f"Khong tim thay anh tai: {INPUT_PATH}. Hay bo anh vao thu muc input/.")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

## Buoc 1: Gaussian Blur (lam mo, giam nhieu)

In [ ]:
blurred = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.4)

## Buoc 2: Tinh Gradient bang toan tu Sobel (do lon + huong)

In [ ]:
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
angle = np.arctan2(sobel_y, sobel_x) * 180 / np.pi  # doi ra do
angle[angle < 0] += 180  # dua ve khoang 0-180

## Buoc 3: Non-Maximum Suppression (lam mong canh)

In [ ]:
def non_max_suppression(magnitude, angle):
    H, W = magnitude.shape
    result = np.zeros((H, W), dtype=np.float32)

    for i in range(1, H - 1):
        for j in range(1, W - 1):
            a = angle[i, j]

            # xac dinh 2 pixel lang gieng theo huong gradient
            if (0 <= a < 22.5) or (157.5 <= a <= 180):
                neighbor1, neighbor2 = magnitude[i, j+1], magnitude[i, j-1]
            elif 22.5 <= a < 67.5:
                neighbor1, neighbor2 = magnitude[i+1, j-1], magnitude[i-1, j+1]
            elif 67.5 <= a < 112.5:
                neighbor1, neighbor2 = magnitude[i+1, j], magnitude[i-1, j]
            else:  # 112.5 <= a < 157.5
                neighbor1, neighbor2 = magnitude[i-1, j-1], magnitude[i+1, j+1]

            # chi giu lai neu la cuc dai cuc bo
            if magnitude[i, j] >= neighbor1 and magnitude[i, j] >= neighbor2:
                result[i, j] = magnitude[i, j]

    return result

nms = non_max_suppression(magnitude, angle)
nms_display = np.uint8(255 * nms / np.max(nms))

## Buoc 4: Hysteresis Thresholding (noi canh manh/yeu)

In [ ]:
def hysteresis_threshold(img, low_ratio=0.05, high_ratio=0.15):
    high_threshold = img.max() * high_ratio
    low_threshold = high_threshold * low_ratio  # (hoac img.max() * low_ratio)

    H, W = img.shape
    result = np.zeros((H, W), dtype=np.uint8)

    strong = 255
    weak = 75

    strong_i, strong_j = np.where(img >= high_threshold)
    weak_i, weak_j = np.where((img >= low_threshold) & (img < high_threshold))

    result[strong_i, strong_j] = strong
    result[weak_i, weak_j] = weak

    # noi canh yeu neu ke can canh manh
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            if result[i, j] == weak:
                if strong in [result[i-1,j-1], result[i-1,j], result[i-1,j+1],
                              result[i,j-1],               result[i,j+1],
                              result[i+1,j-1], result[i+1,j], result[i+1,j+1]]:
                    result[i, j] = strong
                else:
                    result[i, j] = 0

    return result

edges_manual = hysteresis_threshold(nms, low_ratio=0.05, high_ratio=0.15)

## So sanh voi `cv2.Canny()` co san

In [ ]:
edges_opencv = cv2.Canny(blurred, 50, 150)

## Hien thi va luu ket qua tat ca cac buoc

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].imshow(gray, cmap='gray'); axes[0,0].set_title('0. Anh xam goc'); axes[0,0].axis('off')
axes[0,1].imshow(blurred, cmap='gray'); axes[0,1].set_title('1. Gaussian Blur'); axes[0,1].axis('off')
axes[0,2].imshow(np.uint8(255*magnitude/np.max(magnitude)), cmap='gray'); axes[0,2].set_title('2. Sobel Magnitude'); axes[0,2].axis('off')
axes[1,0].imshow(nms_display, cmap='gray'); axes[1,0].set_title('3. Non-Max Suppression'); axes[1,0].axis('off')
axes[1,1].imshow(edges_manual, cmap='gray'); axes[1,1].set_title('4. Hysteresis (tu cai dat)'); axes[1,1].axis('off')
axes[1,2].imshow(edges_opencv, cmap='gray'); axes[1,2].set_title('So sanh: cv2.Canny()'); axes[1,2].axis('off')

plt.tight_layout()
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Da chay xong toan bo 4 buoc. Ket qua da luu tai: {OUTPUT_PATH}')